In [1]:
with open('gsd.txt', encoding='utf-8') as f:
    corpus_raw = f.read()

In [7]:
CORPUS = []
for sent_id, sentence in enumerate(corpus_raw.split('\n\n')):
  for line in sentence.split('\n'):
    if line.startswith('#'):
      continue
    CORPUS.append([sent_id, *line.split('\t')])

In [8]:
df = pd.DataFrame(CORPUS, columns=['sent_id', 'token_id', 'word', 'lemma', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc'])

In [9]:
grams = []
for val in df['feats'].unique():
  if val:
    for cat in val.split('|'):
      grams.append(cat.split('=')[0])
grams = list(set(grams))
grams.remove('_')
# grams

In [12]:
df['feats'] = df['feats'].fillna('_')
df['gram_dict'] = df['feats'].apply(lambda x: {cat.split('=')[0]: cat.split('=')[1] for cat in x.split('|') if '=' in x}) #для категории в иксе сплитанутом по вертикальной черте, если в иксе есть равно, то сделать словарь, где ключ категория до равно

In [13]:
df.head(2)

,sent_id,token_id,word,lemma,upos,xpos,feats,head,deprel,deps,misc,gram_dict
0,0,1,Во,во,ADP,IN,_,2,case,_,_,{}
1,0,2,время,время,NOUN,NN,Animacy=Inan|Case=Acc|Gender=Neut|Number=Sing,18,obl,_,_,"{'Animacy': 'Inan', 'Case': 'Acc', 'Gender': '..."


In [14]:
for gram in grams:
  df[gram] = df['gram_dict'].apply(lambda x: x.get(gram)) # из словаря gram_dict по ключу gram достаем значение, если ключа нет, .get(..) вернет нам None

In [36]:
# теперь можно удалить ненужные столбцы таблицы
df.drop(['feats', 'gram_dict'], axis=1, inplace=True) # axis=1 удаляем колонки

<ipython-input-36-7f2ec040dec8>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(['feats', 'gram_dict'], axis=1, inplace=True) # axis=1 удаляем колонки


In [37]:
# а еще удалим строки, которые не содержат токены
df = df[~df['word'].isna()]

In [38]:
# Простой поиск (без квиков)

In [39]:
# по токенам
token_query = 'стекло'
df[df['word']==token_query].head(5)

,sent_id,token_id,word,lemma,upos,xpos,head,deprel,deps,misc,...,Reflex,Number,Voice,Polarity,Variant,Mood,VerbForm,Degree,NumType,Typo
58978,3005,20,стекло,стекло,NOUN,NN,29,nsubj:pass,_,SpaceAfter=No,...,None,Sing,None,None,None,None,None,None,None,None


In [40]:
# по леммам
lemma_query = 'течь'
df[df['lemma']==lemma_query].head(5)

,sent_id,token_id,word,lemma,upos,xpos,head,deprel,deps,misc,...,Reflex,Number,Voice,Polarity,Variant,Mood,VerbForm,Degree,NumType,Typo
16887,860,13,текущих,течь,VERB,VBNL,11,acl,_,_,...,None,Plur,Act,None,None,None,Part,None,None,None
18962,972,1,Течёт,течь,VERB,VBC,0,root,_,_,...,None,Sing,Act,None,None,Ind,Fin,None,None,None
18978,972,17,течёт,течь,VERB,VBC,1,conj,_,_,...,None,Sing,Act,None,None,Ind,Fin,None,None,None
31901,1631,30,текущих,течь,VERB,VBNL,28,acl,_,_,...,None,Plur,Act,None,None,None,Part,None,None,None
36452,1862,1,Течёт,течь,VERB,VBC,0,root,_,_,...,None,Sing,Act,None,None,Ind,Fin,None,None,None


In [41]:
# по сочетанию чего угодно, условия И задаются через &(), условия ИЛИ задаются через |()
df[
    (df.upos=='VERB')
    & (df.VerbForm=='Fin')
    & (df.Gender=='Masc')
    & (df.Number=='Sing')
]

,sent_id,token_id,word,lemma,upos,xpos,head,deprel,deps,misc,...,Reflex,Number,Voice,Polarity,Variant,Mood,VerbForm,Degree,NumType,Typo
10,0,11,восстановил,восстановить,VERB,VBC,8,acl:relcl,_,_,...,None,Sing,Act,None,None,Ind,Fin,None,None,None
86,4,2,вышел,выйти,VERB,VBC,0,root,_,_,...,None,Sing,Act,None,None,Ind,Fin,None,None,None
105,5,6,переехал,переехать,VERB,VBC,0,root,_,_,...,None,Sing,Act,None,None,Ind,Fin,None,None,None
111,5,12,прожил,прожить,VERB,VBC,8,acl:relcl,_,_,...,None,Sing,Act,None,None,Ind,Fin,None,None,None
155,7,1,Закончил,закончить,VERB,VBC,0,root,_,_,...,None,Sing,Act,None,None,Ind,Fin,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74465,3828,10,участвовал,участвовать,VERB,VBC,6,conj,_,_,...,None,Sing,Act,None,None,Ind,Fin,None,None,None
74753,3841,11,поднялся,подняться,VERB,VBC,7,conj,_,_,...,None,Sing,Mid,None,None,Ind,Fin,None,None,None
74838,3846,24,правил,править,VERB,VBC,21,parataxis,_,_,...,None,Sing,Act,None,None,Ind,Fin,None,None,None
74881,3848,5,входил,входить,VERB,VBC,0,root,_,_,...,None,Sing,Act,None,None,Ind,Fin,None,None,None
